In [1]:
import pandas as pd
import os
import re

from sympy.core.numbers import NaN

# ----------------------------
# SETTINGS
# ----------------------------
UNSC_CSV = "./sc_resolutions_1946_2026.csv"
RESOLUTIONS_CSV = "./ga_resolutions_1946_2019.csv"

# ----------------------------
# LOAD DATA
# ----------------------------
unsc_df = None
if not os.path.isfile(UNSC_CSV):
    print(f"Error: File '{UNSC_CSV}' does not exist.")

else:
    print("File found. Loading...")

    try:
        # Load in chunks to avoid memory issues
        chunksize = 2000
        chunks = []

        for chunk in pd.read_csv(UNSC_CSV, chunksize=chunksize):
            chunks.append(chunk)

        unsc_df = pd.concat(chunks, ignore_index=True)
        print("CSV file loaded successfully.")

    except Exception as e:
        print(f"Error loading CSV file: {e}")


File found. Loading...
CSV file loaded successfully.


In [2]:
df = None
if not os.path.isfile(RESOLUTIONS_CSV):
    print(f"Error: File '{RESOLUTIONS_CSV}' does not exist.")
else:
    print("File found. Loading...")

    try:
        # Load in chunks to avoid memory issues
        chunksize = 2000
        chunks = []

        for chunk in pd.read_csv(RESOLUTIONS_CSV, chunksize=chunksize):
            chunks.append(chunk)

        df = pd.concat(chunks, ignore_index=True)
        print("CSV file loaded successfully.")

    except Exception as e:
        print(f"Error loading CSV file: {e}")

File found. Loading...
CSV file loaded successfully.


In [3]:
# constrain the time to 1946 - 2019
unsc_df['year'] = unsc_df['code'].str.extract(r'\((\d{4})\)').astype(int)

unsc_df = unsc_df[unsc_df['year'] <= 2019]


In [4]:
sc_res = unsc_df['code'].str.replace("S/RES/", "", 1)
sc_res = ' ' + sc_res + ' '

In [5]:

print(sc_res)

318      2503 (2019) 
319      2502 (2019) 
320      2501 (2019) 
321      2500 (2019) 
322      2499 (2019) 
            ...      
2723        5 (1946) 
2724        4 (1946) 
2725        3 (1946) 
2726        2 (1946) 
2727        1 (1946) 
Name: code, Length: 2410, dtype: object


In [6]:
# ----------------------------
# PROCESS RESOLUTIONS
# ----------------------------
res_id2 = []
sc_resolution = []

for sc_res, code in zip(unsc_df['code'], sc_res):
    for res_id, content in zip(df['res_id2'], df['content']):
        if code in content:
            res_id2.append(res_id)
            sc_resolution.append(sc_res)
        if sc_res in content:
            res_id2.append(res_id)
            sc_resolution.append(sc_res)

cites = pd.DataFrame({'res_id2': res_id2, 'sc_resolution': sc_resolution})
cites = cites.drop_duplicates(subset='res_id2', keep='last')

print('Now our dataframe of cites has', len(cites), 'new registers')



Now our dataframe of cites has 2033 new registers


In [7]:
# ----------------------------
# SAVE DATA
# ----------------------------
cites.to_csv("./sc_citations.csv", index=False)